# P3-02 / I3 · Kill-and-resume test (toy model, checkpoints mirrored to Drive)

Trains the P3-02 toy model (`p3_ops.cloud.demo`), kills it, resumes it, and prints a results table.
Pass: the run resumes from its last checkpoint, **at most 1 epoch is lost**, and it finishes.

* **Part A (automatic, ~3 min):** the notebook kills its own training process half-way through epoch `KILL_AT`,
  deletes the local checkpoints (so they must come back from Drive), and resumes. Supporting evidence.
* **Part B (the real I3 test, ~15 min):** you stop the whole Kaggle session while training runs, reopen the
  notebook, and resume. Steps are in `docs/p3_kaggle_runbook.md` section 10.

Kaggle settings: Accelerator **GPU T4** (or None: the toy model runs on CPU too), Internet **On**,
secret **RCLONE_CONF** attached (*Add-ons → Secrets*), and your private dataset with `p3_code_bundle.zip`.

In [ ]:
# Cell 1: settings and helpers
import json, os, re, shutil, subprocess, sys, time
from datetime import date
from pathlib import Path

SMOKE = os.environ.get("P3_NOTEBOOK_SMOKE") == "1"   # set only by the local CPU smoke test, never on Kaggle
TODAY = date.today().isoformat()
RUN_A = f"killtest_A_{TODAY}"       # Part A: in-notebook process kill
RUN_B = f"killtest_B_{TODAY}"       # Part B: real session stop; do NOT change it between the two sessions
EPOCHS = 6 if SMOKE else 20
EPOCH_SECONDS = 1 if SMOKE else 30
KILL_AT = 3 if SMOKE else 4         # Part A kills the run half-way through this epoch
SEED = 0
LAST_PRINTED_B = None               # Part B, after the restart: the last "epoch N" number you saw before stopping


def demo_cmd(run):
    return [sys.executable, "-m", "p3_ops.cloud.demo", "--run", run, "--epochs", str(EPOCHS),
            "--epoch-seconds", str(EPOCH_SECONDS), "--seed", str(SEED)]


def stream(cmd, kill_after_epoch=None):
    """Run cmd, show its output live, return its lines. Optionally hard-kill it half-way into the next epoch."""
    env = {**os.environ, "PYTHONPATH": os.pathsep.join([str(Path("src").resolve()), os.environ.get("PYTHONPATH", "")])}
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    lines = []
    for line in p.stdout:
        print(line, end="", flush=True)
        lines.append(line.rstrip())
        if kill_after_epoch is not None and re.match(rf"^epoch\s+{kill_after_epoch}\b", line.strip()):
            time.sleep(EPOCH_SECONDS / 2)
            p.kill()                                   # SIGKILL: no clean-up, like a dead session
            print(f"\n*** KILLED half-way through epoch {kill_after_epoch + 1} ***", flush=True)
            break
    p.wait()
    return lines


def store_info(lines):
    m = next((re.match(r"env=(\S+) dir=(\S+) remote=(\S+)", l) for l in lines if l.startswith("env=")), None)
    return (m.group(2), None if m.group(3) == "None" else m.group(3)) if m else (None, None)


def drive_listing(remote):
    if not remote or not shutil.which("rclone"):
        return None
    out = subprocess.run(["rclone", "lsf", remote], capture_output=True, text=True)
    return sorted(l for l in out.stdout.split() if l.endswith(".pt"))

In [ ]:
# Cell 2: get the P3 code
import glob
if SMOKE:
    REPO = Path(os.environ["P3_REPO"])
else:
    REPO = Path("/kaggle/working/cv_proj")
    hits = glob.glob("/kaggle/input/**/notebooks/p3/kill_resume_test.ipynb", recursive=True)
    zips = glob.glob("/kaggle/input/**/p3_code_bundle.zip", recursive=True)
    if not REPO.exists() and hits:
        shutil.copytree(Path(hits[0]).parents[2], REPO)
    elif not REPO.exists() and zips:
        shutil.unpack_archive(zips[0], REPO)
    elif not REPO.exists():
        raise SystemExit("P3 code not found: add your private dataset with p3_code_bundle.zip (+ Add Input)")
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
from p3_ops import kill_test_log as ktl
print("P3 code:", REPO)

In [ ]:
# Cell 3: connect Drive (rclone + the RCLONE_CONF secret). Without it the test cannot show Drive recovery.
if not SMOKE:
    if not shutil.which("rclone"):
        subprocess.run("wget -q https://downloads.rclone.org/rclone-current-linux-amd64.zip -O /tmp/rclone.zip "
                       "&& unzip -q -o /tmp/rclone.zip -d /tmp && cp /tmp/rclone-*-linux-amd64/rclone /usr/local/bin/",
                       shell=True, check=True)
    from p3_ops.cloud import setup_rclone_from_kaggle_secret
    print("rclone configured:", setup_rclone_from_kaggle_secret())
try:
    import torch
    GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except ImportError:
    GPU = None
PLATFORM = "local" if SMOKE else "kaggle"
print("platform", PLATFORM, "| GPU", GPU)

## Part A: automatic kill inside the notebook

In [ ]:
# Cell 4: Part A - train, kill half-way through epoch KILL_AT, wipe local checkpoints, resume
before = stream(demo_cmd(RUN_A), kill_after_epoch=KILL_AT - 1)
local_dir, remote = store_info(before)
on_drive = drive_listing(remote)
if remote and local_dir:
    shutil.rmtree(local_dir, ignore_errors=True)     # a dead session loses its disk: resume must come from Drive
    print(f"local checkpoints deleted; Drive has: {on_drive}")
else:
    print("WARNING: no Drive remote, local checkpoints kept (Drive recovery is NOT tested)")
after = stream(demo_cmd(RUN_A))
RESULT_A = ktl.analyse("A", before, after, EPOCHS, ckpts_on_drive=on_drive, platform=PLATFORM, gpu=GPU,
                       run=RUN_A, day=TODAY)
print(ktl.table([RESULT_A]))

## Part B: the real session stop (I3 acceptance)

1. Run **Cell 5**. When the output shows `epoch   4` (or later), stop the session: top-right power button →
   **Stop session**. Note the last `epoch N` line you saw.
2. Open the notebook again (*Edit*), run **Cells 1-3**, set `LAST_PRINTED_B = N` in Cell 1 and run it again,
   then run **Cell 6** and **Cell 7**. Do NOT run Cell 5 again.

In [ ]:
# Cell 5: Part B start - stop the SESSION while this runs (only the smoke test kills it by itself)
B_BEFORE = stream(demo_cmd(RUN_B), kill_after_epoch=KILL_AT - 1 if SMOKE else None)

In [ ]:
# Cell 6: Part B resume - run after the restart (Cells 1-3 first, LAST_PRINTED_B set)
if SMOKE:
    LAST_PRINTED_B = ktl.last_epoch(B_BEFORE)
assert LAST_PRINTED_B is not None, "set LAST_PRINTED_B in Cell 1 to the last epoch you saw before stopping"
probe = subprocess.run([sys.executable, "-c", "from p3_ops.cloud import CheckpointStore as S; "
                        f"s = S.for_run('{RUN_B}'); print(s.remote)"], capture_output=True, text=True,
                       env={**os.environ, "PYTHONPATH": str(Path('src').resolve())})
remote_b = probe.stdout.strip() if probe.stdout.strip() != "None" else None
on_drive_b = drive_listing(remote_b)
print("checkpoints on Drive before resuming:", on_drive_b)
B_AFTER = stream(demo_cmd(RUN_B))
RESULT_B = ktl.analyse("B", None, B_AFTER, EPOCHS, last_printed=LAST_PRINTED_B, ckpts_on_drive=on_drive_b,
                       platform=PLATFORM, gpu=GPU, run=RUN_B, day=TODAY)

**Last cell.** Copy the table and the `KILLTEST_RESULT` lines and paste them back to Claude Code.

In [ ]:
# Cell 7: results to paste back
done = [r for r in (globals().get("RESULT_A"), globals().get("RESULT_B")) if r]
print(ktl.table(done))
for r in done:
    print(ktl.TAG + json.dumps(r))